In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
import pandas as pd
from IPython.display import display

In [19]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

llm = ChatOpenAI(model_name='gpt-3.5-turbo', max_tokens=500)

In [20]:
pdf_link = './assets/anexo-projeto-de-lei.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [21]:
len(pages)

31

In [22]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4_000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)


In [23]:
store = InMemoryStore()

vectorstore = Chroma(embedding_function=embeddings, persist_directory='childVectorDB')

In [24]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [25]:
parent_document_retriever.add_documents(pages, ids=None)

In [26]:
data = parent_document_retriever.vectorstore.get()

df = pd.DataFrame({
    "id": data["ids"],
    "document": data["documents"],
    "metadata": data["metadatas"],
})

display(df)

,id,document,metadata
0,e22a8e74-9e7e-450e-ae47-06f9f79e896d,"PROJETO DE LEI Nº , DE 2023 \nDispõe sob...","{'creationdate': '2023-05-03T18:22:00+00:00', ..."
1,bd7c5e47-e27f-4037-b8fe-3a95275f274b,Dispõe sobre o uso da Inteligência Artificial....,{'doc_id': 'a87df713-7a1b-4f30-bcfc-c88c0ce639...
2,9e0e7933-9649-44c5-ae03-50c653a9289b,CAPÍTULO I \nDISPOSIÇÕES PRELIMINARES \nArt. 1...,{'source': './assets/anexo-projeto-de-lei.pdf'...
3,a010f412-34e5-42d0-8458-2d9cdd704dd7,"o desenvolvimento, implementação e uso respons...","{'moddate': '2023-05-03T19:15:38-03:00', 'tota..."
4,01f54c7f-3440-40bf-a53e-fb1302c851e5,"inteligência artificial (IA) no Brasil, com o ...","{'page': 0, 'producer': 'Aspose.Words for Java..."
...,...,...,...
2368,68d6a860-407c-42a9-8e15-da714a5264aa,máquina e o desenvolvimento de sistema de inte...,"{'page_label': '31', 'total_pages': 31, 'creat..."
2369,2664b32a-b615-410c-ab52-7c2bdba95a00,"contudo, implicar em prejuízo aos titulares de...","{'total_pages': 31, 'creator': 'Microsoft Offi..."
2370,0e2e78ea-9871-447c-9ac5-1ceff1770727,desdobramentos de como a regulação pode foment...,"{'creationdate': '2023-05-03T18:22:00+00:00', ..."
2371,ddae36c4-3f9e-41a4-8967-83054b4c463e,"exposto, e cientes do desafio que a matéria re...","{'start_index': 0, 'total_pages': 31, 'produce..."


In [27]:
TEMPLATE = """
Você é um especialista em legislação e tecnologia. Responda a pergunta abaixo utilizando o contexto informado.

Query:
{query}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [28]:
setup_retrival = RunnableParallel({
    'query': RunnablePassthrough(),
    'context': parent_document_retriever,
})

output_parser = StrOutputParser()

In [29]:
parent_chain_retrival = setup_retrival | rag_prompt | llm | output_parser

In [31]:
from IPython.display import display, Markdown

question = 'Quais os principais riscos do marco legal de IA?'
answer = parent_chain_retrival.invoke(question)

display(Markdown(f"""
---
**Pergunta:** {question}

**Resposta:**

{answer}

---
"""))


---
**Pergunta:** Quais os principais riscos do marco legal de IA?

**Resposta:**

Alguns dos principais riscos do marco legal de IA podem incluir:

1. Violação dos direitos fundamentais: Se o marco legal de IA não conseguir proteger adequadamente os direitos fundamentais das pessoas, poderá haver discriminação, violação da privacidade e outras formas de injustiça.

2. Falta de segurança e confiabilidade dos sistemas de IA: Caso as normas estabelecidas não garantam a implementação de sistemas de IA seguros e confiáveis, podem ocorrer acidentes e danos irreparáveis.

3. Impactos no desenvolvimento científico e tecnológico: Se as regulamentações forem muito restritivas, isso pode limitar a inovação e o progresso tecnológico no campo da Inteligência Artificial.

4. Falta de transparência e responsabilidade: Se o marco legal não exigir transparência nos algoritmos e nas decisões tomadas pelos sistemas de IA, isso pode dificultar a responsabilização em caso de erros ou danos causados pelas máquinas.

Esses são apenas alguns dos possíveis riscos associados ao marco legal de IA, sendo essencial que a legislação seja abrangente e equilibrada para promover o uso responsável e ético da inteligência artificial.

---
